In [5]:
from pathlib import Path
import re

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------
output_dir = Path("/Users/matthewwright/Desktop/26-10-19_check_hard_disks/compare_server_raw_processed")
processed_precip = Path("/Users/matthewwright/Desktop/26-10-19_check_hard_disks/file_list_precip_single_261019_1513.txt")
raw_ops = Path("/Users/matthewwright/Desktop/26-10-19_check_hard_disks/file_list_polar_ops_261019_1513.txt")
raw_ops1 = Path("/Users/matthewwright/Desktop/26-10-19_check_hard_disks/file_list_polar_ops1_261019_1513.txt")

output_dir.mkdir(exist_ok=True, parents=True)

# --------------------------------------------------
# READ FILE LISTS
# --------------------------------------------------

def read_list(path, sur_only=False):
    files = set()
    folders = set()

    # Handle Windows PowerShell UTF-8 / UTF-16 files
    with open(path, "rb") as f:
        header = f.read(4)

    encoding = "utf-16" if header.startswith(
        (b"\xff\xfe", b"\xfe\xff")
    ) else "utf-8-sig"

    with open(path, encoding=encoding) as f:
        for line in f:
            p = line.strip().replace("\\", "/")

            if sur_only and "/sur/" not in f"/{p.lower().strip('/')}/":
                continue

            # Extract path starting from YYYYMMDD
            match = re.search(r"(?i)(?:^|/)(\d{8})(?:/(.*))?$", p)

            if not match:
                continue

            day, remainder = match.groups()

            if remainder is None:
                folders.add(day)
            elif remainder.lower().endswith(".mdv"):
                files.add(f"{day}/{remainder}".lower())
            else:
                folders.add(f"{day}/{remainder}".lower())

    return files, folders


processed_files, processed_folders = read_list(processed_precip)
ops_files, ops_folders = read_list(raw_ops, sur_only=True)
ops1_files, ops1_folders = read_list(raw_ops1, sur_only=True)

# --------------------------------------------------
# COMPARE FILES
# --------------------------------------------------

ops_missing = ops_files - processed_files
ops1_missing = ops1_files - processed_files

both_missing = ops_missing & ops1_missing

ops_only_missing = ops_missing - ops1_missing
ops1_only_missing = ops1_missing - ops_missing

all_missing = ops_missing | ops1_missing

# --------------------------------------------------
# COMPARE FOLDERS
# --------------------------------------------------

ops_folders_missing = ops_folders - processed_folders
ops1_folders_missing = ops1_folders - processed_folders

both_folders_missing = (
    ops_folders_missing & ops1_folders_missing
)

# --------------------------------------------------
# SUMMARY
# --------------------------------------------------

print("FILE COMPARISON")
print("-" * 55)
print(f"Processed files:                {len(processed_files):,}")
print(f"Raw OPS files:                  {len(ops_files):,}")
print(f"Raw OPS1 files:                 {len(ops1_files):,}")
print()
print(f"OPS missing from processed:     {len(ops_missing):,}")
print(f"OPS1 missing from processed:    {len(ops1_missing):,}")
print(f"Both missing from processed:    {len(both_missing):,}")
print()
print(f"OPS only missing:               {len(ops_only_missing):,}")
print(f"OPS1 only missing:              {len(ops1_only_missing):,}")
print(f"TOTAL unique missing files:     {len(all_missing):,}")

print("\nFOLDER COMPARISON")
print("-" * 55)
print(f"OPS folders missing:            {len(ops_folders_missing):,}")
print(f"OPS1 folders missing:           {len(ops1_folders_missing):,}")
print(f"Both folders missing:           {len(both_folders_missing):,}")

# --------------------------------------------------
# SAVE MISSING FILE LISTS
# --------------------------------------------------

results = {
    "ops_missing.txt": ops_missing,
    "ops1_missing.txt": ops1_missing,
    "both_missing.txt": both_missing,
    "ops_only_missing.txt": ops_only_missing,
    "ops1_only_missing.txt": ops1_only_missing,
    "all_missing.txt": all_missing,
    "ops_folders_missing.txt": ops_folders_missing,
    "ops1_folders_missing.txt": ops1_folders_missing,
}

for filename, paths in results.items():
    (output_dir / filename).write_text(
        "\n".join(sorted(paths)),
        encoding="utf-8"
    )

print(f"\nResults saved to: {output_dir}")

FILE COMPARISON
-------------------------------------------------------
Processed files:                89,701
Raw OPS files:                  73,754
Raw OPS1 files:                 289

OPS missing from processed:     73,539
OPS1 missing from processed:    286
Both missing from processed:    286

OPS only missing:               73,253
OPS1 only missing:              0
TOTAL unique missing files:     73,539

FOLDER COMPARISON
-------------------------------------------------------
OPS folders missing:            1,295
OPS1 folders missing:           3
Both folders missing:           3

Results saved to: /Users/matthewwright/Desktop/26-10-19_check_hard_disks/compare_server_raw_processed


In [6]:
# --------------------------------------------------
# PROCESSED FILES NOT IN RAW OPS / OPS1
# --------------------------------------------------

processed_not_ops = processed_files - ops_files
processed_not_ops1 = processed_files - ops1_files

# Not present in either raw dataset
processed_not_either = processed_files - (ops_files | ops1_files)

print("\nPROCESSED FILES MISSING FROM RAW")
print("-" * 55)
print(f"Processed not in OPS/sur:       {len(processed_not_ops):,}")
print(f"Processed not in OPS1/sur:      {len(processed_not_ops1):,}")
print(f"Processed not in either:        {len(processed_not_either):,}")

# Save results
for name, paths in {
    "processed_not_ops.txt": processed_not_ops,
    "processed_not_ops1.txt": processed_not_ops1,
    "processed_not_either.txt": processed_not_either,
}.items():
    (output_dir / name).write_text(
        "\n".join(sorted(paths)),
        encoding="utf-8"
    )


PROCESSED FILES MISSING FROM RAW
-------------------------------------------------------
Processed not in OPS/sur:       89,486
Processed not in OPS1/sur:      89,698
Processed not in either:        89,486


In [7]:
# Extract dates from the existing file sets
processed_days = {p.split("/")[0] for p in processed_files}
ops_days = {p.split("/")[0] for p in ops_files}
ops1_days = {p.split("/")[0] for p in ops1_files}

print("DAILY FOLDER COMPARISON")
print("-" * 45)

print(f"Processed days:                 {len(processed_days):,}")
print(f"OPS days:                       {len(ops_days):,}")
print(f"OPS1 days:                      {len(ops1_days):,}")

print()
print(f"OPS days not processed:         {len(ops_days - processed_days):,}")
print(f"OPS1 days not processed:        {len(ops1_days - processed_days):,}")
print(f"Both raw, not processed:        {len((ops_days & ops1_days) - processed_days):,}")
print(f"Processed days absent from raw: {len(processed_days - (ops_days | ops1_days)):,}")

# Save lists
for name, days in {
    "ops_days_not_processed.txt": ops_days - processed_days,
    "ops1_days_not_processed.txt": ops1_days - processed_days,
    "processed_days_not_raw.txt": processed_days - (ops_days | ops1_days),
}.items():
    (output_dir / name).write_text(
        "\n".join(sorted(days)),
        encoding="utf-8"
    )

DAILY FOLDER COMPARISON
---------------------------------------------
Processed days:                 393
OPS days:                       329
OPS1 days:                      2

OPS days not processed:         151
OPS1 days not processed:        0
Both raw, not processed:        0
Processed days absent from raw: 215
